# Pronósticos basados en series de tiempo
## Suavización Exponencial de Holt-Winters
### Maestría en Ciencia de Datos — Universidad Javeriana Cali
**Estudiante:** Juan Sebastian Torres Romero

---

**Ejercicio:** Empleando la información del número de ocupados en miles de personas (Ocupados) para las 13 principales ciudades, encuentre el mejor pronóstico para los próximos 6 meses. Escriba un breve informe de máximo una página de texto que explique cómo llega a sus proyecciones, presente las proyecciones y aclare las limitaciones.

## **1. Carga de paquetes**

In [ ]:
import sys
!{sys.executable} -m pip install --quiet numpy pandas matplotlib scikit-learn statsmodels openpyxl

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from sklearn.metrics import mean_squared_error

## **2. Carga de datos**

Se carga el archivo `datosEmpleo.xlsx` desde el repositorio del curso. Este contiene series mensuales para las 13 principales ciudades de Colombia: tasa de desempleo, ocupados, desocupados e inactivos. La columna `mes` se define como índice de fechas con frecuencia mensual (`MS`), de manera que cada observación queda correctamente ubicada en el tiempo.

In [ ]:
data = pd.read_excel(
    "https://raw.githubusercontent.com/dagudelo30/Series-de-tiempo---Javeriana-Cali/main/intro-moving_average/datosEmpleo.xlsx",
    index_col='mes', parse_dates=True
)
data = data.asfreq("MS")
data.head()

In [ ]:
print(data.shape)
data.info()

El conjunto de datos contiene **222 observaciones mensuales** sin valores faltantes, lo que garantiza una serie completa y lista para el análisis. La variable de interés es `Ocupados`, que registra el número de personas empleadas (en miles) en las 13 ciudades principales.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["Ocupados"], color="steelblue", linewidth=1.3)
plt.title("Número de ocupados en las 13 principales ciudades (miles de personas)", fontsize=13)
plt.xlabel("Mes")
plt.ylabel("Ocupados (miles)")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

La serie muestra una **tendencia creciente sostenida** entre 2001 y 2019, con el número de ocupados pasando de aproximadamente 6.900 a más de 10.800 miles de personas. Se observan también **fluctuaciones periódicas regulares** a lo largo de cada año, lo que sugiere la presencia de un componente estacional. Esta combinación de tendencia y estacionalidad hace necesario el uso de modelos que puedan capturar ambas dinámicas simultáneamente.

## **3. Métodos de suavización exponencial**

Se reservan las **últimas 6 observaciones** (enero–junio 2019) como muestra de prueba y las 216 anteriores para entrenamiento. Esta partición permite evaluar el desempeño predictivo de cada modelo sobre datos no vistos durante la estimación.

In [ ]:
train_td = data[["Ocupados"]][:-6]
test_td  = data[["Ocupados"]][-6:]

print(f"Entrenamiento: {train_td.index[0].strftime('%Y-%m')} a {train_td.index[-1].strftime('%Y-%m')} (n={len(train_td)})")
print(f"Prueba:        {test_td.index[0].strftime('%Y-%m')} a {test_td.index[-1].strftime('%Y-%m')} (n={len(test_td)})")

plt.figure(figsize=(12, 5))
plt.plot(train_td, label="Entrenamiento", color="steelblue")
plt.plot(test_td,  label="Prueba", color="orange", linewidth=2)
plt.title("División entrenamiento / prueba — Ocupados")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
train_td

In [ ]:
test_td

### **3.1 Suavización Exponencial Simple (SES)**

La SES modela únicamente el **nivel** de la serie, asignando mayor peso a las observaciones más recientes mediante el parámetro α. Al no incorporar tendencia ni estacionalidad, se espera que su desempeño sea limitado para esta serie.

In [ ]:
ses_model  = ETSModel(endog=train_td["Ocupados"], error="add")
ses_result = ses_model.fit(disp=False)

point_forecast_ses = ses_result.forecast(6)
ci_ses   = ses_result.get_prediction(start=point_forecast_ses.index[0], end=point_forecast_ses.index[-1])
preds_ses = ci_ses.summary_frame(alpha=0.05).rename(
    columns={"mean": "Point_forecast", "pi_lower": "lower_95", "pi_upper": "upper_95"})

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_td, label="Entrenamiento", color="steelblue")
plt.plot(test_td,  label="Real (prueba)", color="black", linewidth=2)
plt.plot(preds_ses["Point_forecast"], label="SES", linestyle="--", color="tomato")
plt.fill_between(preds_ses.index, preds_ses["lower_95"], preds_ses["upper_95"], alpha=0.2, color="tomato")
plt.title("Suavización Exponencial Simple (SES) — Ocupados")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
print("Alpha:", round(ses_result.alpha, 4))
rmse_ses = np.sqrt(mean_squared_error(test_td["Ocupados"], point_forecast_ses))
print("RMSE SES:", round(rmse_ses, 2))

Como se observa en la gráfica, la SES genera un **pronóstico prácticamente plano**, incapaz de seguir la tendencia creciente de la serie. El RMSE obtenido confirma este bajo desempeño y establece una línea base para comparar modelos más complejos.

### **3.2 Suavización Exponencial Lineal (Holt)**

El modelo de Holt extiende la SES incorporando un componente de **tendencia**, controlado por el parámetro β. Esto permite que el pronóstico crezca en el tiempo, aunque sin capturar la estacionalidad anual.

In [ ]:
holt_model  = ETSModel(endog=train_td["Ocupados"], error="mul", trend="mul")
holt_result = holt_model.fit(disp=False)

point_forecast_holt = holt_result.forecast(6)
ci_holt   = holt_result.get_prediction(start=point_forecast_holt.index[0], end=point_forecast_holt.index[-1])
preds_holt = ci_holt.summary_frame(alpha=0.05).rename(
    columns={"mean": "Point_forecast", "pi_lower": "lower_95", "pi_upper": "upper_95"})

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_td, label="Entrenamiento", color="steelblue")
plt.plot(test_td,  label="Real (prueba)", color="black", linewidth=2)
plt.plot(preds_holt["Point_forecast"], label="Holt", linestyle="--", color="darkorange")
plt.fill_between(preds_holt.index, preds_holt["lower_95"], preds_holt["upper_95"], alpha=0.2, color="darkorange")
plt.title("Suavización Exponencial Lineal (Holt) — Ocupados")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
print("Alpha:", round(holt_result.alpha, 4))
print("Beta:",  round(holt_result.beta, 4))
rmse_holt = np.sqrt(mean_squared_error(test_td["Ocupados"], point_forecast_holt))
print("RMSE Holt:", round(rmse_holt, 2))

Aunque el modelo Holt incorpora tendencia, su RMSE resulta **mayor que el de la SES**. Esto indica que modelar tendencia sin estacionalidad introduce mayor error en este horizonte de 6 meses, donde las fluctuaciones estacionales son determinantes. El pronóstico sobreestima los valores reales al no descontar los meses de menor ocupación típica.

### **3.3 Holt-Winters Aditivo**

El modelo Holt-Winters aditivo incorpora **tendencia y estacionalidad**, donde la amplitud de las fluctuaciones estacionales se asume constante en el tiempo. El parámetro γ controla la velocidad de actualización del componente estacional.

In [ ]:
hw_add_model  = ETSModel(endog=train_td["Ocupados"], error="add",
                         trend="add", seasonal="add", seasonal_periods=12)
hw_add_result = hw_add_model.fit(disp=False)

point_forecast_hw_add = hw_add_result.forecast(6)
ci_hwa   = hw_add_result.get_prediction(start=point_forecast_hw_add.index[0], end=point_forecast_hw_add.index[-1])
preds_hw_add = ci_hwa.summary_frame(alpha=0.05).rename(
    columns={"mean": "Point_forecast", "pi_lower": "lower_95", "pi_upper": "upper_95"})

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_td, label="Entrenamiento", color="steelblue")
plt.plot(test_td,  label="Real (prueba)", color="black", linewidth=2)
plt.plot(preds_hw_add["Point_forecast"], label="HW Aditivo", linestyle="--", color="green")
plt.fill_between(preds_hw_add.index, preds_hw_add["lower_95"], preds_hw_add["upper_95"], alpha=0.2, color="green")
plt.title("Holt-Winters Aditivo — Ocupados")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
print("Alpha:", round(hw_add_result.alpha, 4))
print("Beta:",  round(hw_add_result.beta, 4))
print("Gamma:", round(hw_add_result.gamma, 4))
rmse_hw_add = np.sqrt(mean_squared_error(test_td["Ocupados"], point_forecast_hw_add))
print("RMSE HW Aditivo:", round(rmse_hw_add, 2))

La incorporación de la estacionalidad produce una **mejora sustancial** frente a los modelos anteriores. El RMSE cae significativamente, confirmando que las fluctuaciones mensuales son un componente determinante para pronosticar la ocupación. El modelo logra reproducir el patrón estacional observado en los datos de prueba.

### **3.4 Holt-Winters Multiplicativo sin tendencia**

En este modelo la estacionalidad es **multiplicativa**, es decir, su amplitud crece proporcionalmente al nivel de la serie. Al omitir la tendencia, el modelo asume que el nivel reciente es suficiente para proyectar el futuro.

In [ ]:
hw_mul_model  = ETSModel(endog=train_td["Ocupados"], error="add",
                         trend=None, seasonal="mul", seasonal_periods=12)
hw_mul_result = hw_mul_model.fit(disp=False)

point_forecast_hw_mul = hw_mul_result.forecast(6)
ci_hwm   = hw_mul_result.get_prediction(start=point_forecast_hw_mul.index[0], end=point_forecast_hw_mul.index[-1])
preds_hw_mul = ci_hwm.summary_frame(alpha=0.05).rename(
    columns={"mean": "Point_forecast", "pi_lower": "lower_95", "pi_upper": "upper_95"})

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_td, label="Entrenamiento", color="steelblue")
plt.plot(test_td,  label="Real (prueba)", color="black", linewidth=2)
plt.plot(preds_hw_mul["Point_forecast"], label="HW Multiplicativo (sin tendencia)", linestyle="--", color="purple")
plt.fill_between(preds_hw_mul.index, preds_hw_mul["lower_95"], preds_hw_mul["upper_95"], alpha=0.2, color="purple")
plt.title("Holt-Winters Multiplicativo sin Tendencia — Ocupados")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
print("Alpha:", round(hw_mul_result.alpha, 4))
print("Gamma:", round(hw_mul_result.gamma, 4))
rmse_hw_mul = np.sqrt(mean_squared_error(test_td["Ocupados"], point_forecast_hw_mul))
print("RMSE HW Multiplicativo (sin tendencia):", round(rmse_hw_mul, 2))

El modelo captura bien la estacionalidad pero al omitir la tendencia **subestima el nivel** de ocupación esperado en los meses de prueba. Su RMSE es mayor que el del modelo aditivo con tendencia, lo que sugiere que el crecimiento estructural de la serie sigue siendo relevante para el horizonte de 6 meses.

### **3.5 Holt-Winters Multiplicativo con tendencia**

Este modelo combina **tendencia aditiva y estacionalidad multiplicativa**, lo que le permite capturar simultáneamente el crecimiento sostenido de la serie y las fluctuaciones estacionales de amplitud creciente. Es la especificación más completa evaluada.

In [ ]:
hw_mul_trend_model  = ETSModel(endog=train_td["Ocupados"], error="add",
                               trend="add", seasonal="mul", seasonal_periods=12)
hw_mul_trend_result = hw_mul_trend_model.fit(disp=False)

point_forecast_hw_mul_trend = hw_mul_trend_result.forecast(6)
ci_hwmt   = hw_mul_trend_result.get_prediction(start=point_forecast_hw_mul_trend.index[0], end=point_forecast_hw_mul_trend.index[-1])
preds_hw_mul_trend = ci_hwmt.summary_frame(alpha=0.05).rename(
    columns={"mean": "Point_forecast", "pi_lower": "lower_95", "pi_upper": "upper_95"})

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_td, label="Entrenamiento", color="steelblue")
plt.plot(test_td,  label="Real (prueba)", color="black", linewidth=2)
plt.plot(preds_hw_mul_trend["Point_forecast"], label="HW Multiplicativo (con tendencia)", linestyle="--", color="crimson")
plt.fill_between(preds_hw_mul_trend.index, preds_hw_mul_trend["lower_95"], preds_hw_mul_trend["upper_95"], alpha=0.2, color="crimson")
plt.title("Holt-Winters Multiplicativo con Tendencia — Ocupados")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
print("Alpha:", round(hw_mul_trend_result.alpha, 4))
print("Beta:",  round(hw_mul_trend_result.beta, 4))
print("Gamma:", round(hw_mul_trend_result.gamma, 4))
rmse_hw_mul_trend = np.sqrt(mean_squared_error(test_td["Ocupados"], point_forecast_hw_mul_trend))
print("RMSE HW Multiplicativo (con tendencia):", round(rmse_hw_mul_trend, 2))

El modelo Holt-Winters multiplicativo con tendencia logra el **menor RMSE de todos los modelos evaluados**, ajustándose con mayor precisión a los datos de prueba. La combinación de tendencia y estacionalidad multiplicativa captura adecuadamente tanto el crecimiento sostenido del empleo como las variaciones propias de cada mes del año.

### **3.6 Comparación de modelos**

In [ ]:
comparacion_modelos = pd.DataFrame({
    "Modelo": [
        "SES",
        "Holt (Lineal)",
        "Holt-Winters Aditivo",
        "Holt-Winters Multiplicativo (sin tendencia)",
        "Holt-Winters Multiplicativo (con tendencia)"
    ],
    "Especificacion ETS": [
        "ETS(A, N, N)", "ETS(M, M, N)",
        "ETS(A, A, A)", "ETS(A, N, M)", "ETS(A, A, M)"
    ],
    "RMSE": [
        round(rmse_ses, 2), round(rmse_holt, 2),
        round(rmse_hw_add, 2), round(rmse_hw_mul, 2),
        round(rmse_hw_mul_trend, 2)
    ]
})

comparacion_modelos.style.format({"RMSE": "{:.2f}"}) \
    .highlight_min(subset=["RMSE"], color="lightgreen")

Se compararon cinco modelos de suavización exponencial. Los resultados muestran que los modelos que incorporan estacionalidad presentan un desempeño significativamente mejor. El **Holt-Winters Multiplicativo con tendencia** obtiene el menor RMSE, siendo seleccionado como el modelo final para el pronóstico.

## **4. Pronóstico con todo el dataset**

El modelo **ETS(A, A, M)** se reestima utilizando la totalidad de las 222 observaciones disponibles, maximizando la información incorporada antes de generar el pronóstico para los próximos 6 meses.

In [ ]:
final_model = ETSModel(
    endog=data["Ocupados"],
    error="add", trend="add", seasonal="mul", seasonal_periods=12
)
final_model_fit = final_model.fit(disp=False)

print(f"Alpha: {final_model_fit.alpha:.4f}")
print(f"Beta:  {final_model_fit.beta:.4f}")
print(f"Gamma: {final_model_fit.gamma:.4f}")

In [ ]:
point_forecast_final = final_model_fit.forecast(6)

ci_final = final_model_fit.get_prediction(
    start=point_forecast_final.index[0],
    end=point_forecast_final.index[-1]
)
ci_final_df = ci_final.summary_frame(alpha=0.05)

preds_final = pd.DataFrame({
    "Point_forecast": ci_final_df["mean"],
    "lower_95":       ci_final_df["pi_lower"],
    "upper_95":       ci_final_df["pi_upper"]
})
preds_final

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["Ocupados"], label="Serie historica", color="steelblue", linewidth=1.2)
plt.plot(preds_final["Point_forecast"],
         label="Pronostico ETS(A,A,M)", color="crimson", linewidth=2, linestyle="--", marker="o")
plt.fill_between(
    preds_final.index,
    preds_final["lower_95"],
    preds_final["upper_95"],
    color="crimson", alpha=0.15, label="IC 95%"
)
plt.axvline(x=data.index[-1], color="gray", linestyle=":", linewidth=1.5)
plt.title("Pronostico de Ocupados — Holt-Winters ETS(A,A,M) — jul-dic 2019", fontsize=13)
plt.xlabel("Mes")
plt.ylabel("Miles de personas")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

El pronóstico muestra un **comportamiento creciente con fluctuaciones estacionales**, coherente con la dinámica histórica observada. Se proyecta un pico en noviembre de 2019 (~11.151 miles), seguido de una leve corrección en diciembre, patrón consistente con el comportamiento típico del mercado laboral colombiano hacia fin de año. Los intervalos de confianza al 95% se amplían progresivamente, reflejando la incertidumbre creciente conforme avanza el horizonte de pronóstico.